# Add controlled variation to the illustrative dataset

> **Why this notebook exists.** The Mancino Market data is synthetic and is being used to demonstrate the dashboard, not to make claims about a real retailer. Its payment mix is almost exactly 25% per method, and the category mix barely changes from store to store. That makes the pies look unnaturally perfect and makes filters feel like they barely do anything.
>
> This notebook creates a reproducible, clearly labeled variation of the source data so that selecting different stores produces visibly different category mixes and payment methods. It changes only each transaction's store and payment-type assignments. Product quantities, line prices, discounts, transaction totals, and the original source files remain intact. The notebook writes a separate `Data/v2_illustrative_variation` folder; it does not alter MySQL or overwrite `Data/v1_2025-11-01`. These are illustrative assumptions, not inferred facts about a real company.

Run the cells from top to bottom. The saved output is a separate input directory that can be selected explicitly when reloading the database.

In [1]:
from pathlib import Path
import shutil

import numpy as np
import pandas as pd

# Find the project root whether Jupyter opened at the repo root or Data Prep.
project_root = Path.cwd().resolve()
while not (project_root / 'Data' / 'v1_2025-11-01').is_dir():
    if project_root == project_root.parent:
        raise FileNotFoundError('Could not find Data/v1_2025-11-01 from the notebook working directory.')
    project_root = project_root.parent

source_dir = project_root / 'Data' / 'v1_2025-11-01'
output_dir = project_root / 'Data' / 'v2_illustrative_variation'
seed = 20261029  # Fixed seed makes every run reproducible.
rng = np.random.default_rng(seed)

stores = pd.read_csv(source_dir / 'stores.csv')
products = pd.read_csv(source_dir / 'products.csv', dtype={'product_id': 'string'})
transactions = pd.read_csv(
    source_dir / 'transactions.csv',
    dtype={'transaction_id': 'string', 'customer_id': 'string'},
)
line_items = pd.read_csv(
    source_dir / 'transaction_line_items.csv',
    dtype={'transaction_id': 'string', 'product_id': 'string'},
)
original_transactions = transactions.copy(deep=True)

print(f'Loaded {len(transactions):,} transactions and {len(line_items):,} line items.')
print(f'Output will be written to: {output_dir}')

Loaded 167,823 transactions and 504,000 line items.
Output will be written to: F:\OneDrive\Academics\bits\OOAD\Project\Data\v2_illustrative_variation


In [2]:
def transaction_mix(transaction_frame, item_frame, product_frame, store_frame):
    category_sales = (
        item_frame.merge(product_frame[['product_id', 'category']], on='product_id', validate='many_to_one')
        .groupby(['transaction_id', 'category'], as_index=False)['line_total'].sum()
    )
    by_store_category = category_sales.merge(
        transaction_frame[['transaction_id', 'store_id']], on='transaction_id', validate='many_to_one'
    ).merge(store_frame[['store_id', 'store_name']], on='store_id', validate='many_to_one')
    totals = by_store_category.pivot_table(
        index='store_name', columns='category', values='line_total', aggfunc='sum', fill_value=0
    )
    return totals.div(totals.sum(axis=1), axis=0).mul(100)

def mix_summary(payment_frame, item_frame, category_share_frame):
    transaction_revenue = item_frame.groupby('transaction_id')['line_total'].sum()
    payment_revenue = payment_frame[['transaction_id', 'payment_type']].merge(
        transaction_revenue.rename('revenue'), on='transaction_id', validate='one_to_one'
    )
    payment_share = (
        payment_revenue.groupby('payment_type')['revenue'].sum()
        .div(payment_revenue['revenue'].sum()).mul(100).sort_values(ascending=False).round(2)
    )
    per_category_store_spread = category_share_frame.std(axis=0).mean()
    largest_category_range = (category_share_frame.max() - category_share_frame.min()).max()
    return payment_share, per_category_store_spread, largest_category_range

before_category_shares = transaction_mix(original_transactions, line_items, products, stores)
before_payment_share, before_mean_spread, before_max_range = mix_summary(
    original_transactions, line_items, before_category_shares
)
print('Original revenue share by payment type (%):')
display(before_payment_share.to_frame('share_percent'))
print(f'Mean across-store category-share standard deviation: {before_mean_spread:.3f} percentage points')
print(f'Largest category share range across stores: {before_max_range:.3f} percentage points')

Original revenue share by payment type (%):


,share_percent
payment_type,
CASH,25.11
CREDIT,25.03
DEBIT,25.01
WALLET,24.86


Mean across-store category-share standard deviation: 0.085 percentage points
Largest category share range across stores: 0.571 percentage points


## 1. Add store-specific category preferences

The notebook estimates each transaction's category basket, then gives each store a seeded set of synthetic category affinities. Transactions are reassigned probabilistically, with a preference to remain at their source store and with the original store-volume distribution used as a prior. This creates noticeable but bounded variation while keeping every transaction and its line items together. The affinity values are invented for dashboard demonstration; they are not claims about the neighborhoods or stores.

In [3]:
category_sales = (
    line_items.merge(products[['product_id', 'category']], on='product_id', validate='many_to_one')
    .groupby(['transaction_id', 'category'], as_index=False)['line_total'].sum()
)
basket = category_sales.pivot(
    index='transaction_id', columns='category', values='line_total'
).reindex(transactions['transaction_id']).fillna(0)
category_names = basket.columns.tolist()
basket_totals = basket.sum(axis=1).replace(0, 1)
basket_shares = basket.div(basket_totals, axis=0).to_numpy(dtype=float)

store_ids = stores['store_id'].to_numpy()
store_priors = (
    original_transactions['store_id'].value_counts(normalize=True)
    .reindex(store_ids).fillna(0).to_numpy(dtype=float)
)
source_store_index = original_transactions['store_id'].map(
    {store_id: index for index, store_id in enumerate(store_ids)}
).to_numpy(dtype=int)

# Randomized synthetic preferences; column normalization avoids a global store advantage.
affinity = rng.lognormal(mean=0.0, sigma=0.48, size=(len(store_ids), len(category_names)))
affinity = affinity / affinity.mean(axis=0, keepdims=True)
utilities = basket_shares @ np.log(affinity).T
utilities += np.log(np.maximum(store_priors, 1e-12))[None, :]
# Keep some inertia so the generated version remains recognizably related to its source.
utilities[np.arange(len(transactions)), source_store_index] += np.log(1.8)
utilities -= utilities.max(axis=1, keepdims=True)
probabilities = np.exp(utilities)
probabilities /= probabilities.sum(axis=1, keepdims=True)

draws = rng.random(len(transactions))
assigned_store_index = (draws[:, None] > probabilities.cumsum(axis=1)).sum(axis=1)
assigned_store_index = np.minimum(assigned_store_index, len(store_ids) - 1)
transactions['store_id'] = store_ids[assigned_store_index]

pd.DataFrame(affinity, index=stores['store_name'], columns=category_names).round(2)

,Alcohol,Bakery,Baking,Beverages,Breakfast & Cereal,Candy,Condiments/Sauces/Spices,Dairy & Eggs,Deli,Frozen Foods,Grains and Pasta,Meat and Seafood,Pet Food,Prepared Meals,Snacks/Cookies/Chips
store_name,,,,,,,,,,,,,,,
Mancino Market - Midtown,0.51,1.85,1.38,1.27,0.89,0.34,0.45,0.99,1.95,0.77,0.57,0.84,0.83,0.79,0.49
Mancino Market - NoHo,1.63,1.08,1.59,0.73,1.60,1.91,0.80,0.44,0.35,0.90,1.06,0.85,1.07,0.90,1.34
Mancino Market - Chelsea,0.75,0.70,0.96,1.51,1.30,1.06,1.52,0.74,0.47,0.59,0.81,1.31,1.51,0.37,1.13
Mancino Market - Greenwich Village,0.59,0.70,0.39,0.50,0.56,0.93,0.71,1.87,1.18,0.97,1.03,1.03,0.81,1.59,0.68
Mancino Market - Nolita,1.52,0.67,0.68,0.99,0.65,0.76,1.52,0.96,1.05,1.77,1.53,0.97,0.79,1.36,1.36


## 2. Add store-specific payment preferences

The target overall payment mix is intentionally uneven (cash and credit are more common than debit, and digital wallet is least common). Each store receives a slightly different probability profile drawn around that target. Payment labels change only; transaction amounts do not. This is a synthetic preference profile chosen to make dashboard comparisons visible.

In [4]:
payment_types = sorted(transactions['payment_type'].dropna().unique().tolist())
target_share = {'CASH': 0.36, 'CREDIT': 0.30, 'DEBIT': 0.21, 'WALLET': 0.13}
target = np.array([target_share[name.upper()] for name in payment_types], dtype=float)

# Dirichlet profiles keep each store near the target while allowing meaningful local differences.
payment_profiles = rng.dirichlet(target * 85, size=len(store_ids))
for store_index, store_id in enumerate(store_ids):
    mask = transactions['store_id'].eq(store_id).to_numpy()
    transactions.loc[mask, 'payment_type'] = rng.choice(
        payment_types, size=int(mask.sum()), p=payment_profiles[store_index]
    )

pd.DataFrame(payment_profiles, index=stores['store_name'], columns=payment_types).round(3)

,CASH,CREDIT,DEBIT,WALLET
store_name,,,,
Mancino Market - Midtown,0.434,0.276,0.215,0.075
Mancino Market - NoHo,0.337,0.324,0.208,0.131
Mancino Market - Chelsea,0.355,0.259,0.152,0.234
Mancino Market - Greenwich Village,0.332,0.345,0.231,0.092
Mancino Market - Nolita,0.382,0.368,0.182,0.067


In [5]:
# Guardrails: same transaction IDs and monetary totals; only store/payment labels may differ.
assert transactions['transaction_id'].equals(original_transactions['transaction_id'])
assert transactions[['subtotal', 'tax', 'total']].equals(original_transactions[['subtotal', 'tax', 'total']])
assert transactions['store_id'].isin(store_ids).all()
assert transactions['payment_type'].isin(payment_types).all()

after_category_shares = transaction_mix(transactions, line_items, products, stores)
after_payment_share, after_mean_spread, after_max_range = mix_summary(
    transactions, line_items, after_category_shares
)
print('Revised revenue share by payment type (%):')
display(after_payment_share.to_frame('share_percent'))
print(f'Mean across-store category-share standard deviation: {after_mean_spread:.3f} percentage points')
print(f'Largest category share range across stores: {after_max_range:.3f} percentage points')
print('Store transaction counts:')
display(transactions['store_id'].map(stores.set_index('store_id')['store_name']).value_counts().to_frame('transactions'))

Revised revenue share by payment type (%):


,share_percent
payment_type,
CASH,36.41
CREDIT,32.53
DEBIT,19.55
WALLET,11.51


Mean across-store category-share standard deviation: 1.211 percentage points
Largest category share range across stores: 8.041 percentage points
Store transaction counts:


,transactions
store_id,
Mancino Market - Nolita,54773
Mancino Market - Greenwich Village,37407
Mancino Market - Chelsea,31139
Mancino Market - NoHo,27226
Mancino Market - Midtown,17278


In [6]:
# Copy the untouched dimensions and line items; save only the changed transaction file anew.
output_dir.mkdir(parents=True, exist_ok=True)
for filename in ('stores.csv', 'products.csv', 'transaction_line_items.csv'):
    shutil.copy2(source_dir / filename, output_dir / filename)
transactions.to_csv(output_dir / 'transactions.csv', index=False)

print(f'Created the varied illustrative dataset in {output_dir}')
print('Original v1 files and the MySQL database were not changed.')
print('The original v1 files and the MySQL database were not changed by this cell.')

Created the varied illustrative dataset in F:\OneDrive\Academics\bits\OOAD\Project\Data\v2_illustrative_variation
Original v1 files and the MySQL database were not changed.
The original v1 files and the MySQL database were not changed by this cell.


## 3. Optionally ingest the varied copy into MySQL

The next cell calls the existing `Data Prep/ingest_data.py` loader with the generated v2 folder. It first asks whether to load now, then requests connection details and a password without displaying the password. If the target tables already contain data, the loader will refuse unless you explicitly type `REPLACE`. That choice clears and reloads the four project tables; the original v1 CSVs are still preserved.

In [8]:
import getpass
import os
import subprocess
import sys

load_now = input('Load the varied dataset into MySQL now? Type LOAD to continue: ').strip().upper() == 'LOAD'
if not load_now:
    print('Ingestion skipped. The generated CSV copy is ready at:', output_dir)
else:
    mysql_host = input('MySQL host [localhost]: ').strip() or 'localhost'
    mysql_port = input('MySQL port [3306]: ').strip() or '3306'
    mysql_user = input('MySQL user [root]: ').strip() or 'root'
    mysql_password = getpass.getpass(f'MySQL password for {mysql_user}@{mysql_host}: ')
    replace_existing = input(
        'If target tables already contain data, type REPLACE to clear and reload them; otherwise press Enter: '
    ).strip().upper() == 'REPLACE'

    loader = project_root / 'Data Prep' / 'ingest_data.py'
    command = [
        sys.executable, str(loader), '--host', mysql_host, '--port', mysql_port,
        '--user', mysql_user, '--data-dir', str(output_dir),
    ]
    if replace_existing:
        command.append('--replace')
    environment = os.environ.copy()
    environment['MYSQL_PASSWORD'] = mysql_password
    subprocess.run(command, cwd=project_root, env=environment, check=True)
    print('MySQL ingestion completed.')

MySQL ingestion completed.
